In [1]:
import requests
import pandas as pd

In [2]:
BACKEND_URL = "http://localhost:5000"

def check_backend_health():
    try:
        response = requests.get(f"{BACKEND_URL}/api/health", timeout=5)
        return response.status_code == 200
    except:
        return False

def get_tower_data():
    try:
        response = requests.get(f"{BACKEND_URL}/api/towers", timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data['success']:
                return pd.DataFrame(data['data'])
        return pd.DataFrame()
    except Exception as e:
        # st.error(f"Error fetching tower data: {str(e)}")
        return pd.DataFrame()

def get_predicted_outage_data():
    try:
        response = requests.get(f"{BACKEND_URL}/api/predicted-outages", timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data['success']:
                return pd.DataFrame(data['data'])
        return pd.DataFrame()
    except Exception as e:
        # st.error(f"Error fetching predicted outage data: {str(e)}")
        return pd.DataFrame()


In [3]:
check_backend_health()

True

In [ ]:
outage_data = get_predicted_outage_data()
outage_data.head()

,center_latitude,center_longitude,event,radius,severity
0,33.239560,-103.545897,Cyber Attack,38.514297,Medium
1,29.888784,-99.290398,Flood,48.537938,Low
2,34.085979,-105.670376,Flood,19.662141,Medium
3,32.473680,-95.740795,Wildfire,21.670687,Medium
4,26.441773,-98.288363,Flood,26.130012,Medium


In [5]:
tower_data = get_tower_data()
tower_data.head()

,bandwidth,connected_devices,coverage_radius,data_usage_gb,installation_date,latitude,longitude,signal_strength,status,technology,tower_id,uptime_percentage
0,80,477,2.611971,2677.450580,2024-06-20T14:49:14.973808,33.239560,-94.688451,-73.355710,Active,5G,FN-1000,97.219714
1,100,480,1.178028,3292.655366,2023-01-11T14:49:14.973808,29.888784,-97.403807,-61.690414,Down,4G LTE,FN-1001,99.679687
2,80,516,1.314170,2500.580245,2023-09-06T14:49:14.973808,34.085979,-103.043691,-89.955529,Active,5G,FN-1002,99.269922
3,20,492,4.581966,4850.647665,2024-12-11T14:49:14.973808,32.473680,-93.900707,-111.366150,Active,5G,FN-1003,96.611078
4,100,516,0.700921,6864.257741,2023-02-25T14:49:14.973808,26.441773,-96.376238,-119.163823,Active,5G,FN-1004,97.722742


In [ ]:
import folium
from IPython.display import display

m = folium.Map(location=(31.96, -99.9), zoom_start=6, tiles="cartodb positron")

severity_2_color = {
    'Critical': 'darkred',
    'High': 'red',
    'Medium': 'orange',
    'Low': 'Yellow'
}

status_2_color = {
    'Active': 'green',
    'Down': 'red'
}

for _, row in tower_data.iterrows():
    if row['status'] != 'Down':
        continue
    
    popup_text = ""
    popup_text += f"Tower ID: {row['tower_id']}<br>"
    popup_text += f"Status: {row['status']}<br>"
    popup_text += f"Signal Strength: {row['signal_strength']:.3f} dBm<br>"
    popup_text += f"Bandwidth: {row['bandwidth']} MHz<br>"
    popup_text += f"Technology: {row['technology']}<br>"
    popup_text += f"Lon: {row['longitude']:.4f}<br>"
    popup_text += f"Lat: {row['latitude']:.4f}<br>"
    
    popup = folium.Popup(popup_text, max_width=120)
    
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        tooltip=row['tower_id'],
        popup=popup,
        icon=folium.Icon(color=status_2_color[row['status']])
    ).add_to(m)

for _, row in outage_data.iterrows():
    popup_text = ""
    popup_text += f"Event: {row['event']}<br>"
    popup_text += f"Severity: {row['severity']}<br>"
    popup_text += f"Lon: {row['center_longitude']:.4f}<br>"
    popup_text += f"Lat: {row['center_latitude']:.4f}<br>"
    popup_text += f"Radius: {row['radius']:.2f} km<br>"
    
    popup = folium.Popup(popup_text, max_width=120)
    
    folium.Circle(
        location=[row['center_latitude'], row['center_longitude']],
        radius=row['radius'] * 1000,  # Convert to meters
        color=severity_2_color[row['severity']],
        fill=True,
        fill_color=severity_2_color[row['severity']],
        tooltip=row['event'],
        popup=popup
    ).add_to(m)

display(m)